In [ ]:
# Pipeline parameters — overridden by job base_parameters when run via DAB.
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "dev")
dbutils.widgets.text("volume_name", "raw_files")
dbutils.widgets.text("bronze_write_mode", "overwrite")
dbutils.widgets.text("overwrite_schema", "true")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
bronze_write_mode = dbutils.widgets.get("bronze_write_mode")
overwrite_schema = dbutils.widgets.get("overwrite_schema").lower() == "true"
volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"

print(f"catalog={catalog}  schema={schema}  volume_path={volume_path}")
print(f"bronze_write_mode={bronze_write_mode}  overwrite_schema={overwrite_schema}")

# Bronze Layer — Ingest CSV Files

## Step 3 - Ingest CSV files into Bronze Layer


In [ ]:
from pyspark.sql import functions as F

# ── 1. Discover all CSV files in the volume ───────────────────────────────────
all_files = dbutils.fs.ls(volume_path)
csv_files = [f for f in all_files if f.name.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {volume_path}")

print(f"Found {len(csv_files)} file(s) — creating one bronze table per file:\n")

created_tables = []

# ── 2. One table per file ─────────────────────────────────────────────────────
for file_info in csv_files:
    # Derive a clean table name: bronze_<filename_without_extension>
    base_name  = file_info.name.rsplit(".", 1)[0]
    table_name = "bronze_" + base_name.lower().replace("-", "_").replace(" ", "_")
    full_name  = f"{catalog}.{schema}.{table_name}"

    # Read CSV — preserve ALL original columns (keys, FKs, etc.) for referential integrity
    df = (
        spark.read
             .option("header",      "true")
             .option("inferSchema", "true")
             .csv(file_info.path)
             # Audit columns
             .withColumns({
                 "ingestion_timestamp": F.current_timestamp(),
                 "source_file_name":    F.col("_metadata.file_name")
             })
    )

    # Write as a managed Delta table (CREATE OR REPLACE — idempotent)
    writer = (
        df.write
          .format("delta")
          .mode(bronze_write_mode)
    )
    if overwrite_schema:
        writer = writer.option("overwriteSchema", "true")
    writer.saveAsTable(full_name)

    # Capture column names once to avoid repeated Analyze RPC calls
    col_names = df.columns
    row_count = spark.table(full_name).count()
    created_tables.append({"table": full_name, "source": file_info.name, "rows": row_count, "cols": col_names})

    print(f"  Table   : {full_name}")
    print(f"  Source  : {file_info.name}")
    print(f"  Columns : {', '.join(col_names)}")
    print(f"  Rows    : {row_count:,}")
    print()

# ── 3. Summary ────────────────────────────────────────────────────────────────
print(f"{'─'*60}")
print(f"Bronze layer ready — {len(created_tables)} table(s) created in {catalog}.{schema}")
print(f"{'─'*60}")
for t in created_tables:
    print(f"  {t['table']:50s}  ({t['rows']:,} rows)")